# Atlas v1 — first genuine DP622 benchmark

Atlas reconstructs an **active-like** DP622–Aβ model, evaluates the published mutation controls with genuine ThermoMPNN/ThermoMPNN-D predictions and catalytic geometry, and permits computational candidate generation only if the predefined gate passes. Nothing in this notebook is experimental validation.

## How to run this notebook

1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU**.
3. Run the configuration and hardware-check cells.
4. Confirm the full preflight reports `passed: true`.
5. Run the remaining cells in order.
6. Do not rerun ThermoMPNN or ThermoMPNN-D cells unnecessarily; valid checkpoints are reused automatically.
7. Find the benchmark decision in `validation_report.md` and all machine-readable outputs in the printed run directory. The last cell downloads a ZIP.

## Configuration — review this before running

In [ ]:
ATLAS_REPO_URL = 'https://github.com/Noelduval/atlas-therapeutic-optimization.git'
ATLAS_REF = 'codex/atlas-v1-dynamic-geometry'
THERMOMPNN_REV = '2b04fd370e399911b1fa5848112cc9013f084110'
THERMOMPNN_D_REV = 'df9a75aaddb674a7c4c193005031fc0536d325fb'
DYNAMICS_MODE = 'minimize'
USE_GOOGLE_DRIVE = True  # Recommended: checkpoints survive runtime restarts.
print('Configured Atlas ref:', ATLAS_REF)
print('OpenMM mode:', DYNAMICS_MODE)

## Stage 1 — fast T4 hardware check

In [ ]:
import subprocess
def run_bootstrap_command(stage_name, command, cwd=None):
    import os
    import shlex
    working_directory = os.path.abspath(cwd or os.getcwd())
    exact_command = shlex.join(str(part) for part in command)
    completed = subprocess.run(
        [str(part) for part in command],
        cwd=working_directory,
        text=True,
        capture_output=True,
        check=False,
    )
    if completed.returncode:
        print('\n=== NOTEBOOK SETUP STAGE FAILED ===')
        print('Stage:', stage_name)
        print('Exit status:', completed.returncode)
        print('Working directory:', working_directory)
        print('Exact command:', exact_command)
        print('--- complete stdout ---')
        print(completed.stdout or '<empty>')
        print('--- complete stderr ---')
        print(completed.stderr or '<empty>')
        print('Suggested next action: fix the first setup error above, then rerun this cell.')
        raise RuntimeError(f'{stage_name} failed with exit status {completed.returncode}')
    if completed.stdout:
        print(completed.stdout, end='' if completed.stdout.endswith('\n') else '\n')
    if completed.stderr:
        print(completed.stderr, end='' if completed.stderr.endswith('\n') else '\n')
    return completed

print('Checking the GPU before installing or loading scientific models...')
gpu = run_bootstrap_command(
    'T4 hardware check',
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free,memory.used', '--format=csv,noheader'],
)
print(gpu.stdout.strip())
if 'T4' not in gpu.stdout:
    raise RuntimeError('This reviewer path is configured for a Tesla T4. Select a T4 GPU runtime and reconnect.')
import torch
print('PyTorch:', torch.__version__, '| CUDA runtime:', torch.version.cuda, '| CUDA visible:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch cannot see CUDA. Change the runtime to T4 GPU, reconnect, and rerun.')
print('PyTorch device:', torch.cuda.get_device_name(0))

## Stage 2 — exact repository setup and persistent checkpoint directory

In [ ]:
import hashlib
import os
from pathlib import Path
import subprocess
import sys

ATLAS_DIR = Path('/content/Atlas')
if not (ATLAS_DIR / '.git').exists():
    run_bootstrap_command('Clone Atlas', ['git', 'clone', '--no-checkout', ATLAS_REPO_URL, str(ATLAS_DIR)])
run_bootstrap_command('Fetch configured Atlas ref', ['git', '-C', str(ATLAS_DIR), 'fetch', 'origin', ATLAS_REF])
run_bootstrap_command('Check out configured Atlas ref', ['git', '-C', str(ATLAS_DIR), 'checkout', '--detach', 'FETCH_HEAD'])
ATLAS_SHA = run_bootstrap_command(
    'Resolve Atlas commit',
    ['git', '-C', str(ATLAS_DIR), 'rev-parse', 'HEAD'],
).stdout.strip()
print('Resolved Atlas commit:', ATLAS_SHA)

print('Installing Atlas, OpenMM, and official-model Python dependencies...')
run_bootstrap_command('Install Atlas and scientific Python dependencies', [
    sys.executable, '-m', 'pip', 'install', '--quiet',
    f'{ATLAS_DIR}[dynamics]',
    'omegaconf', 'wandb', 'pytorch-lightning', 'scipy', 'scikit-learn', 'joblib',
    'tqdm', 'torchmetrics',
])

EXTERNAL = ATLAS_DIR / '.external'
EXTERNAL.mkdir(exist_ok=True)
model_repositories = [
    ('https://github.com/Kuhlman-Lab/ThermoMPNN.git', EXTERNAL / 'ThermoMPNN', THERMOMPNN_REV),
    ('https://github.com/Kuhlman-Lab/ThermoMPNN-D.git', EXTERNAL / 'ThermoMPNN-D', THERMOMPNN_D_REV),
]
for url, path, revision in model_repositories:
    if not (path / '.git').exists():
        run_bootstrap_command(f'Clone {path.name}', ['git', 'clone', '--no-checkout', url, str(path)])
    run_bootstrap_command(f'Fetch pinned {path.name}', ['git', '-C', str(path), 'fetch', 'origin', revision])
    run_bootstrap_command(f'Check out pinned {path.name}', ['git', '-C', str(path), 'checkout', '--detach', 'FETCH_HEAD'])
    actual = run_bootstrap_command(
        f'Resolve {path.name} commit',
        ['git', '-C', str(path), 'rev-parse', 'HEAD'],
    ).stdout.strip()
    if actual != revision:
        raise RuntimeError(f'Wrong checkout for {path.name}: {actual}')
    print(f'{path.name} commit: {actual}')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/Atlas/checkpoints')
else:
    OUTPUT_ROOT = ATLAS_DIR / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
INPUT_STRUCTURE = ATLAS_DIR / 'data/23WN.cif'
input_sha = hashlib.sha256(INPUT_STRUCTURE.read_bytes()).hexdigest()
RUN_ID = f'atlas-t4-{ATLAS_SHA[:12]}-{input_sha[:8]}'
RUN_DIR = OUTPUT_ROOT / RUN_ID
os.chdir(ATLAS_DIR)
print('Checkpoint run directory:', RUN_DIR)

## Configure documented upstream runtime paths

In [ ]:
from atlas.colab import configure_upstream_runtime_paths
configured_paths = configure_upstream_runtime_paths(
    EXTERNAL / 'ThermoMPNN',
    EXTERNAL / 'ThermoMPNN-D',
)
for configured_path in configured_paths:
    print('Configured upstream checkout path:', configured_path)

## Full preflight — stop here unless every check passes

In [ ]:
import json
from atlas.colab import StageExecutionError, build_stage_command, run_stage_command
preflight_path = OUTPUT_ROOT / f'preflight-{ATLAS_SHA[:12]}.json'
preflight_command = [
    sys.executable, '-m', 'atlas', 'preflight',
    '--input', str(INPUT_STRUCTURE),
    '--atlas-repo', str(ATLAS_DIR),
    '--thermompnn-repo', str(EXTERNAL / 'ThermoMPNN'),
    '--thermompnn-d-repo', str(EXTERNAL / 'ThermoMPNN-D'),
    '--output-json', str(preflight_path),
]
print('Running the complete lightweight preflight...')
run_stage_command(
    'Complete Atlas environment preflight',
    preflight_command,
    ATLAS_DIR,
    suggested_next_action='Fix the reported preflight error before starting reconstruction.',
)
preflight = json.loads(preflight_path.read_text())
print(json.dumps(preflight, indent=2))
if not preflight['passed']:
    raise RuntimeError('Preflight did not pass. Do not start model inference.')
def run_atlas_stage(label, stop_after=None):
    resume = (RUN_DIR / 'run_context.json').exists()
    command = build_stage_command(
        python_executable=sys.executable,
        input_structure=INPUT_STRUCTURE,
        output_root=OUTPUT_ROOT,
        atlas_repo=ATLAS_DIR,
        thermompnn_repo=EXTERNAL / 'ThermoMPNN',
        thermompnn_d_repo=EXTERNAL / 'ThermoMPNN-D',
        run_id=RUN_ID,
        dynamics_mode=DYNAMICS_MODE,
        stop_after=stop_after,
        resume=resume,
    )
    run_stage_command(
        label,
        command,
        ATLAS_DIR,
        suggested_next_action=(
            'Read the first Atlas error above, resolve it, then rerun this stage. '
            'Do not continue to downstream scientific stages.'
        ),
    )

## Runtime readiness — final cheap checks before Stage 3

In [ ]:
from atlas.colab import validate_colab_readiness
print('Checking Atlas imports, CLI entrypoint, repository/model layout, and checkpoint writability...')
runtime_readiness = validate_colab_readiness(
    python_executable=sys.executable,
    atlas_repo=ATLAS_DIR,
    input_structure=INPUT_STRUCTURE,
    thermompnn_repo=EXTERNAL / 'ThermoMPNN',
    thermompnn_d_repo=EXTERNAL / 'ThermoMPNN-D',
    output_root=OUTPUT_ROOT,
    run_dir=RUN_DIR,
)
print(json.dumps(runtime_readiness, indent=2))

## Stage 3 — active-like reconstruction

In [ ]:
run_atlas_stage('Reconstructing DP622 and writing benchmark PDBs', 'structure')
print('Reconstruction:', RUN_DIR / 'DP622_active_like_reconstruction.pdb')
print('Residue map:', RUN_DIR / 'residue_numbering_map.csv')

## Stage 4 — inspect benchmark preparation

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
display(pd.read_csv(RUN_DIR / 'known_mutants_manifest.csv'))
display(pd.read_csv(RUN_DIR / 'residue_numbering_map.csv').head())

## Stage 5 — ThermoMPNN singles (expensive; checkpointed)

In [ ]:
run_atlas_stage('Running genuine ThermoMPNN for Y91F, D126A, and H172A', 'thermompnn')
display(pd.read_csv(RUN_DIR / 'thermompnn_scores.csv'))
print('This subprocess has exited; its GPU allocations are released.')

## Stage 6 — ThermoMPNN-D epistatic double (expensive; checkpointed)

In [ ]:
run_atlas_stage('Running genuine ThermoMPNN-D for Y91F/D126A', 'thermompnn-d')
display(pd.read_csv(RUN_DIR / 'thermompnn_scores.csv'))
print('This subprocess has exited; its GPU allocations are released.')

## Stage 7 — catalytic geometry

In [ ]:
run_atlas_stage('Calculating deterministic catalytic geometry', 'geometry')
display(pd.read_csv(RUN_DIR / 'geometry_metrics.csv'))

## Stage 8 — restrained OpenMM minimization

In [ ]:
run_atlas_stage('Attempting restrained OpenMM minimization', 'dynamics')
display(pd.read_csv(RUN_DIR / 'openmm_dynamics_summary.csv'))
print('Skipped rows are unavailable evidence, never favorable evidence.')

## Stage 9 — predefined validation gate

In [ ]:
print('Evaluating the fixed gate. Thresholds are not changed after predictions are seen.')
try:
    run_atlas_stage('Evaluating published controls', 'validation')
except StageExecutionError as error:
    if (RUN_DIR / 'known_mutation_validation.csv').exists():
        display(pd.read_csv(RUN_DIR / 'known_mutation_validation.csv'))
    if (RUN_DIR / 'validation_report.md').exists():
        display(Markdown((RUN_DIR / 'validation_report.md').read_text()))
    raise RuntimeError('BENCHMARK FAILED or EXTERNALLY BLOCKED. Candidate generation remains blocked.') from error
display(pd.read_csv(RUN_DIR / 'known_mutation_validation.csv'))
display(Markdown((RUN_DIR / 'validation_report.md').read_text()))
print((RUN_DIR / 'execution_status.json').read_text())

## Stage 10 — conditional candidate generation
Run this only after Stage 9 completes without an exception. The same production pipeline enforces the gate again before creating candidates.

In [ ]:
run_atlas_stage('Running conditional candidate generation and ranking')
ranked = RUN_DIR / 'novel_candidates_ranked.csv'
if not ranked.exists():
    raise RuntimeError('No legitimate candidate ranking was produced.')
display(pd.read_csv(ranked).head(20))
display(Image(filename=str(RUN_DIR / 'figures/candidate_ranking_summary.png')))

## Stage 11 — report and export

In [ ]:
import shutil
display(Markdown((RUN_DIR / 'validation_report.md').read_text()))
display(Markdown((RUN_DIR / 'pipeline_warnings.md').read_text()))
for figure in sorted((RUN_DIR / 'figures').glob('*.png')):
    display(Image(filename=str(figure)))
archive = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Final report:', RUN_DIR / 'validation_report.md')
print('Machine-readable status:', RUN_DIR / 'execution_status.json')
print('Checkpointed ZIP:', archive)
from google.colab import files
files.download(archive)